# Imports

In [11]:
from openmeteo_requests import Client as OpenMeteoClient
from requests_cache import CachedSession
from retry_requests import retry
import pandas as pd 
import requests
import datetime
import time

# Adding weather informations by date

## Constants

In [12]:
BASE_URL_WEATHER = 'https://archive-api.open-meteo.com/v1/archive'
BASE_URL_IP = 'http://ip-api.com/batch?fields=query,status,lat,lon'

PATH_SPOTIFY = '../../data/2_processed/final_df.csv'
PATH_IPS = '../../data/2_processed/ip_addresses.csv'
PATH_WEATHER = '../../data/2_processed/weather.csv'

## Methods

### Address/IP

In [13]:
def get_address_by_list_ips(list_ips):
    url = BASE_URL_IP
    dict_address = []
    
    try:
        response = requests.post(url, json=list_ips)
        response.raise_for_status()
        results = response.json()
        
        for info in results:
            if info.get('status') == 'success':
                ip = info.get('query')
                lat, lon = float(info.get('lat')), float(info.get('lon'))
                dict_address.append({ip:f'{lat:.1f},{lon:.1f}'})
            else:
                print(f"Not able to locate IP: {info.get('query')}")
        
    except requests.exceptions.RequestException as e:
        print(f'Request error: {e}')
        for i in list_ips:
            dict_address.append({i:None})

    return dict_address

In [14]:
def set_address_by_ip(df):
    all_ips = list(set(df.ip_addr))
    all_address = []
    batch_ips = []
    count_request = 0
    for i in all_ips:
        batch_ips.append(i)
        if (len(batch_ips) >= 99):
            all_address.extend(get_address_by_list_ips(batch_ips))
            batch_ips = []
            count_request += 1
            if (count_request >= 14):
                print('Wait 1 minute before making the next call.')
                time.sleep(60)
                count_request = 0

    return all_address

### Weather

In [15]:
def get_weather_history(batch_lat, batch_lon, data_inicio, data_fim):
    cache_session = CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = OpenMeteoClient(session=retry_session)

    params = {
        'latitude': batch_lat, 'longitude': batch_lon,
        'start_date': data_inicio, 'end_date': data_fim,
        'hourly': ['temperature_2m', 'precipitation'],
    }

    responses = openmeteo.weather_api(BASE_URL_WEATHER, params=params)

    all_dfs = []
    for r in responses:
        hourly = r.Hourly()
        dates = pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit='s', utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit='s', utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive='left',
        )
        all_dfs.append(pd.DataFrame({
            'date': dates,
            'temperature': hourly.Variables(0).ValuesAsNumpy(),
            'precipitation': hourly.Variables(1).ValuesAsNumpy(),
            'lat_long': f'{r.Latitude():.1f},{r.Longitude():.1f}',
        }))

    return pd.concat(all_dfs)

In [16]:
def get_all_weather(df):
    date_start = df.iloc[0].ts.split('T')[0]
    date_end = df.iloc[-1].ts.split('T')[0]

    DEFAULT_COORDS = (-23.5471, -46.6372)  # São Paulo
    lats, lons = zip(*(
        tuple(map(float, addr.split(','))) if str(addr).upper() != 'NAN' else DEFAULT_COORDS
        for addr in set(df['lat_long'])
    ))

    all_weather, req_count = [], 0

    for i in range(0, len(lats), 50):
        batch_lat, batch_lon = list(lats[i:i+50]), list(lons[i:i+50])
        all_weather.append(get_weather_history(batch_lat, batch_lon, date_start, date_end))

        if req_count >= 19:
            req_count = 0
            print('Waiting 1 minute for more api calls.')
            time.sleep(60)
        else:
            req_count += 1

    return pd.concat(all_weather)


In [17]:
def optimize_weather_merge(main_df, weather_df):
    # Ensure 'ts' is datetime
    main_df["ts"] = pd.to_datetime(main_df["ts"])

    # Build temporary join keys
    main_df["date"] = main_df["ts"].dt.strftime("%Y-%m-%d")
    main_df["hour"] = main_df["ts"].dt.hour

    # Left-join weather data on location + date + hour
    result_df = (
        main_df
        .merge(
            weather_df[["lat_long", "date", "hour", "temperature", "precipitation"]],
            on=["lat_long", "date", "hour"],
            how="left",
        )
        .drop(columns=["date", "hour"])  # drop temporary join keys
    )

    return result_df

## Creating address df

In [18]:
final_df = pd.read_csv(PATH_SPOTIFY)

In [19]:
address_df = final_df.copy()
all_address = set_address_by_ip(address_df)

In [20]:
df = pd.DataFrame([
    {'ip': list(d.keys())[0], 'lat_long': list(d.values())[0]} 
    for d in all_address
])
ips_address_df = pd.merge(address_df, df, how='left', left_on='ip_addr', right_on='ip').drop(columns='ip')
ips_address_df

,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,...,key,liveness,loudness,mode,speechiness,tempo,valence,popularity,duration_ms,lat_long
0,2020-10-20T19:36:53Z,"Android OS 10 API 29 (samsung, SM-A307GT)",23600,BR,177.58.181.120,Pretty Savage,THE ALBUM,spotify:track:1XnpzbOGptRwfJhZgLbmSr,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3"
1,2020-10-20T19:36:55Z,"Android OS 10 API 29 (samsung, SM-A307GT)",1052,BR,177.58.181.120,How You Like That,THE ALBUM,spotify:track:4SFknyjLcyTLJFPKD2m96o,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3"
2,2020-10-20T19:36:56Z,"Android OS 10 API 29 (samsung, SM-A307GT)",0,BR,177.58.181.120,Ice Cream (with Selena Gomez),THE ALBUM,spotify:track:4JUPEh2DVSXFGExu4Uxevz,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3"
3,2020-10-20T19:36:56Z,"Android OS 10 API 29 (samsung, SM-A307GT)",0,BR,177.58.181.120,Bet You Wanna (feat. Cardi B),THE ALBUM,spotify:track:7iAgNZdotu40NwtoIWJHFe,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3"
4,2020-10-20T19:36:59Z,"Android OS 10 API 29 (samsung, SM-A307GT)",1824,BR,177.58.181.120,Lovesick Girls,THE ALBUM,spotify:track:4Ws314Ylb27BVsvlZOy30C,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7386,2026-01-22T20:51:55Z,ios,201760,BR,2804:7f0:b2c1:40d5:9512:4b9f:c178:1ce0,Hungarian Dance No. 5 (Brahms),Brahms: Hungarian Dances for string orchestra,spotify:track:2dvrarYDL3flRy9ESkqvKK,NaN,NaN,...,7.0,0.0909,-17.594,0.0,0.0505,144.351,0.3490,27.0,201760.0,"-23.6,-46.6"
7387,2026-01-22T20:56:53Z,ios,296853,BR,2804:7f0:b2c1:40d5:9512:4b9f:c178:1ce0,Friska from Hungarian Rhapsody No. 2,Autobahn [Drive Time],spotify:track:3B84LG8ayDrYnfwlm11Vn4,NaN,NaN,...,7.0,0.1260,-13.779,1.0,0.0463,79.711,0.3600,24.0,296853.0,"-23.6,-46.6"
7388,2026-01-23T03:10:40Z,ios,437017,BR,2804:7f0:b2c1:40d5:38c4:1dd8:42d3:546d,Toccata and Fugue in D minor,Great Organ Classics,spotify:track:0BWJNm4TrO6H3qgiCmDBjM,NaN,NaN,...,2.0,0.0937,-25.350,0.0,0.0373,71.749,0.0714,52.0,567466.0,"-23.6,-46.6"
7389,2026-01-23T13:13:29Z,ios,11685,BR,2804:38a:a08e:d9a5:7dbf:86aa:23b2:662f,Tis so Sweet to Trust in Jesus,Lullaby Hymns: The Weary Soul,spotify:track:21fL1tPW9PN7mgqhgeL7lC,NaN,NaN,...,0.0,0.0957,-16.239,1.0,0.0323,80.077,0.1580,39.0,256653.0,"-23.5,-46.6"


In [21]:
ips_address_df.to_csv(PATH_IPS)

## Creating weather df

In [22]:
weather_df = pd.read_csv(PATH_IPS).drop(columns='Unnamed: 0')
display(weather_df)

,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,...,key,liveness,loudness,mode,speechiness,tempo,valence,popularity,duration_ms,lat_long
0,2020-10-20T19:36:53Z,"Android OS 10 API 29 (samsung, SM-A307GT)",23600,BR,177.58.181.120,Pretty Savage,THE ALBUM,spotify:track:1XnpzbOGptRwfJhZgLbmSr,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3"
1,2020-10-20T19:36:55Z,"Android OS 10 API 29 (samsung, SM-A307GT)",1052,BR,177.58.181.120,How You Like That,THE ALBUM,spotify:track:4SFknyjLcyTLJFPKD2m96o,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3"
2,2020-10-20T19:36:56Z,"Android OS 10 API 29 (samsung, SM-A307GT)",0,BR,177.58.181.120,Ice Cream (with Selena Gomez),THE ALBUM,spotify:track:4JUPEh2DVSXFGExu4Uxevz,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3"
3,2020-10-20T19:36:56Z,"Android OS 10 API 29 (samsung, SM-A307GT)",0,BR,177.58.181.120,Bet You Wanna (feat. Cardi B),THE ALBUM,spotify:track:7iAgNZdotu40NwtoIWJHFe,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3"
4,2020-10-20T19:36:59Z,"Android OS 10 API 29 (samsung, SM-A307GT)",1824,BR,177.58.181.120,Lovesick Girls,THE ALBUM,spotify:track:4Ws314Ylb27BVsvlZOy30C,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7386,2026-01-22T20:51:55Z,ios,201760,BR,2804:7f0:b2c1:40d5:9512:4b9f:c178:1ce0,Hungarian Dance No. 5 (Brahms),Brahms: Hungarian Dances for string orchestra,spotify:track:2dvrarYDL3flRy9ESkqvKK,NaN,NaN,...,7.0,0.0909,-17.594,0.0,0.0505,144.351,0.3490,27.0,201760.0,"-23.6,-46.6"
7387,2026-01-22T20:56:53Z,ios,296853,BR,2804:7f0:b2c1:40d5:9512:4b9f:c178:1ce0,Friska from Hungarian Rhapsody No. 2,Autobahn [Drive Time],spotify:track:3B84LG8ayDrYnfwlm11Vn4,NaN,NaN,...,7.0,0.1260,-13.779,1.0,0.0463,79.711,0.3600,24.0,296853.0,"-23.6,-46.6"
7388,2026-01-23T03:10:40Z,ios,437017,BR,2804:7f0:b2c1:40d5:38c4:1dd8:42d3:546d,Toccata and Fugue in D minor,Great Organ Classics,spotify:track:0BWJNm4TrO6H3qgiCmDBjM,NaN,NaN,...,2.0,0.0937,-25.350,0.0,0.0373,71.749,0.0714,52.0,567466.0,"-23.6,-46.6"
7389,2026-01-23T13:13:29Z,ios,11685,BR,2804:38a:a08e:d9a5:7dbf:86aa:23b2:662f,Tis so Sweet to Trust in Jesus,Lullaby Hymns: The Weary Soul,spotify:track:21fL1tPW9PN7mgqhgeL7lC,NaN,NaN,...,0.0,0.0957,-16.239,1.0,0.0323,80.077,0.1580,39.0,256653.0,"-23.5,-46.6"


In [23]:
just_weather_df = get_all_weather(weather_df)

In [24]:
all_datetimes = just_weather_df['date']
all_dates, all_hours = [], []
for i in all_datetimes:
    all_dates.append(i.strftime('%Y-%m-%d'))
    all_hours.append(i.hour)

just_weather_df['date'] = all_dates
just_weather_df['hour'] = all_hours
just_weather_df

,date,temperature,precipitation,lat_long,hour
0,2020-10-20,19.000000,0.0,"-23.5,-46.6",0
1,2020-10-20,18.600000,0.0,"-23.5,-46.6",1
2,2020-10-20,18.200001,0.3,"-23.5,-46.6",2
3,2020-10-20,18.150000,0.1,"-23.5,-46.6",3
4,2020-10-20,18.150000,0.3,"-23.5,-46.6",4
...,...,...,...,...,...
46147,2026-01-24,-4.100000,0.1,"32.8,-96.8",19
46148,2026-01-24,-3.900000,0.2,"32.8,-96.8",20
46149,2026-01-24,-3.750000,0.2,"32.8,-96.8",21
46150,2026-01-24,-3.200000,0.0,"32.8,-96.8",22


In [25]:
just_weather_df.to_csv(PATH_WEATHER)
just_weather_df

,date,temperature,precipitation,lat_long,hour
0,2020-10-20,19.000000,0.0,"-23.5,-46.6",0
1,2020-10-20,18.600000,0.0,"-23.5,-46.6",1
2,2020-10-20,18.200001,0.3,"-23.5,-46.6",2
3,2020-10-20,18.150000,0.1,"-23.5,-46.6",3
4,2020-10-20,18.150000,0.3,"-23.5,-46.6",4
...,...,...,...,...,...
46147,2026-01-24,-4.100000,0.1,"32.8,-96.8",19
46148,2026-01-24,-3.900000,0.2,"32.8,-96.8",20
46149,2026-01-24,-3.750000,0.2,"32.8,-96.8",21
46150,2026-01-24,-3.200000,0.0,"32.8,-96.8",22


## Creating final df with weather

In [26]:
just_weather_df = pd.read_csv(PATH_WEATHER)

final_weather_df = optimize_weather_merge(weather_df, just_weather_df)
final_weather_df

,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,...,loudness,mode,speechiness,tempo,valence,popularity,duration_ms,lat_long,temperature,precipitation
0,2020-10-20 19:36:53+00:00,"Android OS 10 API 29 (samsung, SM-A307GT)",23600,BR,177.58.181.120,Pretty Savage,THE ALBUM,spotify:track:1XnpzbOGptRwfJhZgLbmSr,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3",23.00,0.8
1,2020-10-20 19:36:55+00:00,"Android OS 10 API 29 (samsung, SM-A307GT)",1052,BR,177.58.181.120,How You Like That,THE ALBUM,spotify:track:4SFknyjLcyTLJFPKD2m96o,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3",23.00,0.8
2,2020-10-20 19:36:56+00:00,"Android OS 10 API 29 (samsung, SM-A307GT)",0,BR,177.58.181.120,Ice Cream (with Selena Gomez),THE ALBUM,spotify:track:4JUPEh2DVSXFGExu4Uxevz,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3",23.00,0.8
3,2020-10-20 19:36:56+00:00,"Android OS 10 API 29 (samsung, SM-A307GT)",0,BR,177.58.181.120,Bet You Wanna (feat. Cardi B),THE ALBUM,spotify:track:7iAgNZdotu40NwtoIWJHFe,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3",23.00,0.8
4,2020-10-20 19:36:59+00:00,"Android OS 10 API 29 (samsung, SM-A307GT)",1824,BR,177.58.181.120,Lovesick Girls,THE ALBUM,spotify:track:4Ws314Ylb27BVsvlZOy30C,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3",23.00,0.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10887,2026-01-22 20:56:53+00:00,ios,296853,BR,2804:7f0:b2c1:40d5:9512:4b9f:c178:1ce0,Friska from Hungarian Rhapsody No. 2,Autobahn [Drive Time],spotify:track:3B84LG8ayDrYnfwlm11Vn4,NaN,NaN,...,-13.779,1.0,0.0463,79.711,0.3600,24.0,296853.0,"-23.6,-46.6",17.55,0.3
10888,2026-01-23 03:10:40+00:00,ios,437017,BR,2804:7f0:b2c1:40d5:38c4:1dd8:42d3:546d,Toccata and Fugue in D minor,Great Organ Classics,spotify:track:0BWJNm4TrO6H3qgiCmDBjM,NaN,NaN,...,-25.350,0.0,0.0373,71.749,0.0714,52.0,567466.0,"-23.6,-46.6",16.10,0.0
10889,2026-01-23 13:13:29+00:00,ios,11685,BR,2804:38a:a08e:d9a5:7dbf:86aa:23b2:662f,Tis so Sweet to Trust in Jesus,Lullaby Hymns: The Weary Soul,spotify:track:21fL1tPW9PN7mgqhgeL7lC,NaN,NaN,...,-16.239,1.0,0.0323,80.077,0.1580,39.0,256653.0,"-23.5,-46.6",21.20,0.4
10890,2026-01-23 13:13:29+00:00,ios,11685,BR,2804:38a:a08e:d9a5:7dbf:86aa:23b2:662f,Tis so Sweet to Trust in Jesus,Lullaby Hymns: The Weary Soul,spotify:track:21fL1tPW9PN7mgqhgeL7lC,NaN,NaN,...,-16.239,1.0,0.0323,80.077,0.1580,39.0,256653.0,"-23.5,-46.6",21.20,0.4


In [27]:
final_weather_df.to_csv(PATH_SPOTIFY)